# Story 3: Parallel Computations on pandas DataFrames
You keep the inputs of a screening study - sample IDs, compositions and temperatures - in a
[pandas](https://pandas.pydata.org) DataFrame and compute a new property for every row. You want to replace the slow
`df.apply()` by a parallel version, move large tables to the cluster, and compute only the new rows whenever rows are
added to the table.

**With plain SLURM** you split the DataFrame into files, write a batch script or job array which reads one part per
job, computes it and writes a result file, and finally stitch the parts back together in the original order. Keeping
track of which rows were already computed is up to you.

**With executorlib** `map()` and `submit()` accept DataFrame columns and whole DataFrames as arguments and return
Python objects, so rows or chunks of rows are distributed without any files. The `SlurmClusterExecutor` submits one
SLURM job per chunk, you join the returned DataFrames with `pandas.concat()`, and the cache skips chunks which were
computed before.

This notebook runs on the **login node** of a SLURM cluster.

## Cluster-specific settings
**Adapt the following cell to your cluster.** It collects everything which depends on the cluster rather than on your science. The values below are the ones used to test this notebook on a single-node SLURM test cluster. The submission template is a [Jinja2](https://jinja.palletsprojects.com) template which executorlib renders for every function call: add the lines your cluster requires, for example `#SBATCH --partition=<name>`, `#SBATCH --account=<project>` or `module load ...` commands before `{{command}}`. The [executorlib-cluster-setup](https://executorlib.readthedocs.io/en/latest/2-hpc-cluster.html#setup-with-claude-code) skill for Claude Code can analyse your cluster and generate a tested template.

In [ ]:
# Cluster-specific settings - adapt these values to your cluster.
submission_template = """\
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --job-name={{job_name}}
#SBATCH --chdir={{working_directory}}
#SBATCH --get-user-env
#SBATCH --ntasks={{cores}}
{%- if run_time_max %}
#SBATCH --time={{ [1, run_time_max // 60]|max }}
{%- endif %}
{%- if dependency_list %}
#SBATCH --dependency=afterok:{{ dependency_list | join(',') }}
{%- endif %}

{{command}}
"""

## Step 1: The table and the serial version
The property is computed by a cheap model - 50 milliseconds per row - which takes two columns of the table as input.
The serial version uses `df.apply()` with `axis=1` to call it once per row.

In [ ]:
import time

import numpy
import pandas


def compute_property(composition, temperature):
    # Stand-in for a cheap model evaluated for every row, e.g. a fitted energy model.
    time.sleep(0.05)
    return -3.0 * composition + 0.4 * composition**2 + 1.0e-4 * temperature * (1.0 - composition)


df = pandas.DataFrame(
    {
        "sample_id": numpy.arange(13),
        "composition": numpy.linspace(0.0, 0.6, 13),
        "temperature": numpy.tile([300, 600, 900], 5)[:13],
    }
)
df_serial = df.assign(
    energy=df.apply(lambda row: compute_property(row["composition"], row["temperature"]), axis=1)
)
df_serial.head()

## Step 2: Row-wise parallelism with `map()`
`map()` of the [Executor interface](https://docs.python.org/3/library/concurrent.futures.html#concurrent.futures.Executor.map)
iterates over several columns at once, just like the built-in `map()`, and returns the results in the order of the
rows - so the list can be assigned directly as a new column.

Each row only takes 50 milliseconds. This is far less than the overhead of a SLURM job, so the rows are distributed
with the `SingleNodeExecutor` over the cores of the machine this notebook runs on - your workstation, or a compute
node when the notebook runs inside an allocation. On a shared login node keep this to light work like this small
example. `block_allocation=True` starts the two worker processes once and reuses them for all rows, instead of
starting a new Python process per row.

In [ ]:
from executorlib import SingleNodeExecutor

with SingleNodeExecutor(max_workers=2, block_allocation=True) as exe:
    energy_lst = list(exe.map(compute_property, df["composition"], df["temperature"]))

df_rowwise = df.assign(energy=energy_lst)
numpy.allclose(df_rowwise["energy"], df_serial["energy"])

When individual rows need different resources, or you want to react to each result as it arrives, use `submit()`
with one future per row instead:

```python
future_lst = [exe.submit(compute_property, row.composition, row.temperature) for row in df.itertuples()]
df_rowwise = df.assign(energy=[future.result() for future in future_lst])
```

## Step 3: Chunks of rows as SLURM jobs
For large tables the work moves to the cluster - but not row by row. Every task has an overhead: for the
`SingleNodeExecutor` it is milliseconds, for an `srun` job step of the `SlurmJobExecutor` inside an allocation around
a second, and for a SLURM job of the `SlurmClusterExecutor` it is the time in the queue plus starting Python and
reading and writing the cache - seconds to hours. In addition many clusters limit the number of jobs per user. So the
rows are grouped into chunks, and every chunk becomes one task:

* **Too fine:** the overhead dominates and the queue is flooded. For SLURM jobs choose chunks which run for minutes to
  hours.
* **Too coarse:** fewer chunks than available cores limit the parallelism, and a failing chunk means recomputing all
  of its rows.

A chunk does not have to use a single core: submit it with `threads_per_core` and process its rows with a
`SingleNodeExecutor` inside the function, as shown in [Story 10](story-10-fan-out-inside-jobs.ipynb).

The chunks are fixed-size slices of the table taken with `iloc`. `numpy.array_split(df, n)` would split the table into a
fixed *number* of chunks instead - but then the chunk boundaries move whenever rows are added, so every chunk changes
and none of them can be reloaded from the cache in Step 4. With fixed-size slices, appending rows only changes the
last chunk. If the rows belong to natural groups, `df.groupby(column)` is the other good choice: one chunk per group,
and only the groups which receive new rows change.

Each chunk is submitted with a `cache_key` computed from its content by `get_chunk_key()` - Step 4 explains why.

In [ ]:
import hashlib


def process_chunk(chunk):
    # Executed as one SLURM job: compute the property for all rows of the chunk.
    energy_lst = [compute_property(c, t) for c, t in zip(chunk["composition"], chunk["temperature"])]
    return chunk.assign(energy=energy_lst)


def split_into_chunks(df, chunk_size):
    # Fixed-size chunks: appending rows only changes the last chunk and adds new ones.
    return [df.iloc[start : start + chunk_size] for start in range(0, len(df), chunk_size)]


def get_chunk_key(chunk):
    # Name the cache files of a chunk after its content: values, index, column names and dtypes.
    content_hash = hashlib.md5(pandas.util.hash_pandas_object(chunk, index=True).to_numpy().tobytes())
    content_hash.update(str(chunk.dtypes.to_dict()).encode())
    return "process_chunk_v1_" + content_hash.hexdigest()

Every chunk is submitted as its own SLURM job with one core, the returned DataFrames are joined with `pandas.concat()`.
As `concat()` keeps the index of every chunk and the futures are collected in submission order, the result has the
rows in the original order.

In [ ]:
from executorlib import SlurmClusterExecutor

cache_directory = "./story10_cache"
chunk_size = 5  # rows per SLURM job - in production rather thousands of rows


def compute_in_chunks(df, chunk_size):
    with SlurmClusterExecutor(
        cache_directory=cache_directory,
        resource_dict={"submission_template": submission_template, "run_time_max": 300},
    ) as exe:
        future_lst = [
            exe.submit(process_chunk, chunk, resource_dict={"cache_key": get_chunk_key(chunk)})
            for chunk in split_into_chunks(df, chunk_size)
        ]
        return pandas.concat([future.result() for future in future_lst])


df_chunked = compute_in_chunks(df, chunk_size)
numpy.allclose(df_chunked["energy"], df_serial["energy"])

## Step 4: New rows arrive - compute only what is new
Four new samples are appended to the table with `pandas.concat()`.

By default executorlib identifies a function call by a hash of the pickled function and its arguments. For a DataFrame
the pickle does not only contain the values, the index, the column names and the dtypes, but also the internal memory
layout of pandas - which columns are stored together in one block. `pandas.concat()` can change this layout, so the
first chunk of the extended table may get a new default hash although its content is identical, and every chunk would
be computed again. The check below compares the content with `equals()` and the pickled bytes which the default hash
is based on - depending on your pandas version the second value can be `False`.

`get_chunk_key()` avoids this: [`pandas.util.hash_pandas_object()`](https://pandas.pydata.org/docs/reference/api/pandas.util.hash_pandas_object.html)
hashes the values and the index row by row, independent of the memory layout, and the column names and dtypes are
added to the hash. Chunks with the same rows, the same index labels and the same columns get the same key and are
reloaded from the cache. A changed value, a different index (for example after `reset_index()` or sorting) or a
changed dtype leads to a new key.

In [ ]:
import cloudpickle

new_rows = pandas.DataFrame(
    {"sample_id": numpy.arange(13, 17), "composition": [0.65, 0.7, 0.75, 0.8], "temperature": [600, 900, 300, 600]}
)
df_extended = pandas.concat([df, new_rows], ignore_index=True)

first_chunk_before = split_into_chunks(df, chunk_size)[0]
first_chunk_after = split_into_chunks(df_extended, chunk_size)[0]
{
    "same content": first_chunk_before.equals(first_chunk_after),
    "same pickle (default hash)": cloudpickle.dumps(first_chunk_before) == cloudpickle.dumps(first_chunk_after),
    "same cache_key": get_chunk_key(first_chunk_before) == get_chunk_key(first_chunk_after),
}

The extended table is split into the chunks with rows 0-4, 5-9, 10-14 and 15-16. The first two are identical to
chunks computed in Step 3 and are loaded from the cache, only the last two are submitted as new SLURM jobs. The
previous last chunk with rows 10-12 is no longer requested - its result stays in the cache, but it is not used.

An explicit `cache_key` replaces the default hash completely, so it no longer changes when the code of
`process_chunk()` changes. This is why the key contains a version, `v1`: increase it whenever you change the
computation, otherwise the old results are reloaded. [Story 8](story-8-recover-failed-jobs.ipynb) uses the same
property on purpose to fix a bug without recomputing the successful calls.

In [ ]:
from executorlib import get_cache_data

df_extended_result = compute_in_chunks(df_extended, chunk_size)
len(df_extended_result), len(get_cache_data(cache_directory=cache_directory))

## Step 5: Collect the results from the cache
The cache can be read without an executor, for example from a new Python process the next day. `get_cache_data()`
returns one entry per cached call with its input arguments, its output and its run time. The first argument of every
call is the chunk, so its index shows which rows were computed by which job.

In [ ]:
df_cache = pandas.DataFrame(get_cache_data(cache_directory=cache_directory))
df_cache["first_row"] = df_cache["input_args"].apply(lambda args: args[0].index[0])
df_cache["last_row"] = df_cache["input_args"].apply(lambda args: args[0].index[-1])
df_cache[["first_row", "last_row", "runtime"]].sort_values(["first_row", "last_row"])

The outputs are DataFrames, so they are joined with `pandas.concat()`. The superseded chunk with rows 10-12 overlaps
with the chunk with rows 10-14 - both computed the same values, so duplicated index labels are dropped.

In [ ]:
df_all = pandas.concat(df_cache["output"].tolist())
df_all = df_all[~df_all.index.duplicated()].sort_index()
numpy.allclose(df_all["energy"], df_extended_result["energy"]), len(df_all)

## Clean up

In [ ]:
import shutil

shutil.rmtree(cache_directory, ignore_errors=True)

## What you learned
* `map()` over DataFrame columns replaces `df.apply()`; the results come back in the order of the rows and are
  assigned as a new column.
* Cheap rows are grouped into chunks - fixed-size `iloc` slices or `groupby()` groups - submitted as one task each and
  joined with `pandas.concat()`. Choose the chunk size so that the work per task outweighs its overhead: minutes or
  more for a SLURM job.
* The default cache key hashes the pickled DataFrame, including its internal memory layout. A `cache_key` derived from
  `pandas.util.hash_pandas_object()` makes an extended table reuse the finished chunks.
* `get_cache_data()` collects the cached chunks into one DataFrame without an executor.

**Reference:** [HPC Cluster Executor](https://executorlib.readthedocs.io/en/latest/2-hpc-cluster.html),
[cache](https://executorlib.readthedocs.io/en/latest/1-single-node.html#cache),
[block allocation](https://executorlib.readthedocs.io/en/latest/1-single-node.html#block-allocation) and the
[resource dictionary](https://executorlib.readthedocs.io/en/latest/trouble_shooting.html#resource-dictionary).